<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/17-Using_LLMs_to_rank_chunks_as_the_Judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [2]:
!pip install -q huggingface-hub==0.34.4 llama-index==0.13.3 llama-index-llms-openai==0.5.4 llama-index-vector-stores-chroma==0.5.2 \
                llama-index-llms-google-genai==0.3.0 chromadb==1.0.20 jedi==0.19.2

In [3]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "[OPENAI_API_KEY]"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')

# Load a Model


In [4]:
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


# Load Indexes


Downloading vector store from Huggingface hub

In [5]:
from huggingface_hub import hf_hub_download
vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip",repo_type="dataset",local_dir="/content")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [6]:
!unzip /content/vectorstore.zip

Archive:  /content/vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [7]:
# Load the vector store from the local storage.
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db2 = chromadb.PersistentClient(path="/content/ai_tutor_knowledge")
chroma_collection = db2.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [8]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding

# Create the index based on the vector store.
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

In [9]:
query_engine = index.as_query_engine(similarity_top_k=10)

res = query_engine.query("Explain how Advance RAG works?")

In [10]:
res.response

'The excerpts do not mention anything called "Advance RAG." They describe several RAG variants and related ideas—notably Adaptive RAG (often called Adaptive or routing RAG), Corrective RAG (fallback/evaluator-based), Self‑Reflective / Self‑RAG (self‑correction via reflection tags), Speculative RAG (draft‑and‑verify with a small drafter + large verifier), and standard RAG formulations (conditioning on same passages vs. per‑token passages).\n\nIf you meant one of those (for example "Adaptive RAG" or "Advanced RAG"), tell me which and I’ll explain how that approach works using the provided material.'

# RankGPT


In [11]:
from llama_index.core.postprocessor.rankGPT_rerank import RankGPTRerank

rankGPT = RankGPTRerank(top_n=3,llm=Settings.llm, verbose=True)

In [12]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
# The `node_postprocessors` function will be applied to the retrieved nodes.
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[rankGPT])

res = query_engine.query("Explain how Retrieval Augmented Generation (RAG) works?")

After Reranking, new rank list for nodes: [3, 1, 0, 4, 6, 8, 9, 7, 2, 5]

In [15]:
res.response

'Retrieval-Augmented Generation (RAG) is a hybrid approach that combines an information retrieval subsystem with a generative language model to improve performance on knowledge-intensive tasks. The process can be described in these main parts and steps:\n\n1. Two complementary memories\n- Non‑parametric memory: an external document store (e.g., Wikipedia) organized into an index. Documents are represented so they can be efficiently retrieved (sparse inverted indexes or dense vector embeddings).\n- Parametric memory: a pretrained sequence-to-sequence (seq2seq) generative model (or large language model) that produces the final text output.\n\n2. Retrieval component\n- Indexing: source documents are preprocessed and organized into an index suited to the chosen retrieval method (sparse or dense).\n- Searching: given a user query, the retriever finds a set of relevant passages or documents from the index. Dense retrieval uses vector encodings and nearest-neighbor search; sparse retrieval us

In [14]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text.strip())
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 e791eaf2-3cdf-477a-8602-1cbb30a3d48c
Title	 RAG
Text	 # RAG<div class="flex flex-wrap space-x-1"><a href="https://huggingface.co/models?filter=rag"><img alt="Models" src="https://img.shields.io/badge/All_model_pages-rag-blueviolet"></a></div>## OverviewRetrieval-augmented generation ("RAG") models combine the powers of pretrained dense retrieval (DPR) andsequence-to-sequence models. RAG models retrieve documents, pass them to a seq2seq model, then marginalize to generateoutputs. The retriever and seq2seq modules are initialized from pretrained models, and fine-tuned jointly, allowingboth retrieval and generation to adapt to downstream tasks.It is based on the paper [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401) by Patrick Lewis, Ethan Perez, Aleksandara Piktus, Fabio Petroni, VladimirKarpukhin, Naman Goyal, Heinrich Küttler, Mike Lewis, Wen-tau Yih, Tim Rocktäschel, Sebastian Riedel, Douwe Kiela.The abstract from the pape

# RankGPT — Quick Notes

## Definition
RankGPT is an **LLM-based reranker** that reorders retrieved chunks by **semantic relevance using reasoning**, before passing them to the generator in a RAG pipeline.

---

## Role in RAG

Query  
↓  
Dense Retriever (top-k)  
↓  
RankGPT Reranker  
↓  
Top-n refined chunks  
↓  
LLM Generator

```yaml
# example flow (illustrative)
query: "user query"
retriever: dense_retriever(top_k=10)
reranker: RankGPT(top_n=3)
generator: LLM
```

---

## Core Function

- Input: query + retrieved chunks  
- LLM reasons over all chunks  
- Outputs a **new ranking order**  
- Keeps only `top_n` best chunks

---

## Why Use RankGPT

- Fixes mis-ranked embedding results  
- Improves rank-1 precision  
- Increases faithfulness and grounding  
- Reduces hallucinations

---

## Trade-offs

| Aspect   | Impact |
|----------|--------|
| Accuracy | Very high |
| Latency  | High |
| Cost     | High |
| Scale    | Limited |

Not suitable for high-QPS or low-latency systems.

---

## Code Pattern (LlamaIndex)

```python
from llama_index.core.postprocessor.rankGPT_rerank import RankGPTRerank

rankGPT = RankGPTRerank(top_n=3, llm=Settings.llm)

query_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[rankGPT]
)
```

---

## Comparison

| Method           | Accuracy | Speed  | Reasoning      |
|------------------|----------|--------|----------------|
| Dense retriever  | Medium   | Fast   | No             |
| Cross-encoder    | High     | Medium | Limited        |
| RankGPT          | Highest  | Slow   | Full           |

---

## Best Use Cases

- Knowledge-intensive QA  
- Technical / legal / medical RAG  
- Evaluation pipelines  
- Premium accuracy tiers

---

## Key Takeaway
RankGPT = LLM-powered semantic reranking — highest accuracy layer in RAG, used selectively due to cost and latency.

# Custom Postprocessor


## The `Judger` Function


The following function will query GPT-5 (mini) to retrieve the top three nodes that has highest similarity to the asked question.


In [16]:
from pydantic import BaseModel
from llama_index.llms.openai import OpenAI
from llama_index.core.prompts import PromptTemplate


def judger(nodes, query):

    # The model's output template
    class OrderedNodes(BaseModel):
        """A node with the id and assigned score."""

        node_id: list
        score: list

    # Prepare the nodes and wrap them in <NODE></NODE> identifier, as well as the query
    the_nodes = ""
    for idx, item in enumerate(nodes):
        the_nodes += f"<NODE{idx+1}>\nNode ID: {item.node_id}\nText: {item.text}\n</NODE{idx+1}>\n"

    query = "<QUERY>\n{}\n</QUERY>".format(query)

    # Define the prompt template
    prompt_tmpl = PromptTemplate(
        """
    You receive a qurey along with a list of nodes' text and their ids. Your task is to assign score
    to each node based on its contextually closeness to the given query. The final output is each
    node id along with its proximity score.
    Here is the list of nodes:
    {nodes_list}

    And the following is the query:
    {user_query}

    Score each of the nodes based on their text and their relevancy to the provided query.
    The score must be a decimal number between 0 an 1 so we can rank them."""
    )

    # Define the an instance of GPT-5 mini and send the request
    llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})
    ordered_nodes = llm.structured_predict(
        OrderedNodes, prompt_tmpl, nodes_list=the_nodes, user_query=query
    )

    return ordered_nodes

## Define Postprocessor


The following class will use the `judger` function to rank the nodes, and filter them based on the ranks.


In [17]:
from typing import List, Optional
from llama_index.core import QueryBundle
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore


class OpenaiAsJudgePostprocessor(BaseNodePostprocessor):
    def _postprocess_nodes(
        self, nodes: List[NodeWithScore], query_bundle: Optional[QueryBundle]
    ) -> List[NodeWithScore]:

        r = judger(nodes, query_bundle)

        node_ids = r.node_id
        scores = r.score

        sorted_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        num_nodes_to_select = min(3, len(sorted_scores))
        top_nodes = [sorted_scores[i][0] for i in range(num_nodes_to_select)]
        # top_nodes = [sorted_scores[i][0] for i in range(3)]

        selected_nodes_id = [node_ids[item] for item in top_nodes]

        final_nodes = []
        for item in nodes:
            if item.node_id in selected_nodes_id:
                final_nodes.append(item)

        return final_nodes

In [18]:
judge = OpenaiAsJudgePostprocessor()

## Query Engine with Postprocessor


In [19]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
# The `node_postprocessors` function will be applied to the retrieved nodes.
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[judge])

res = query_engine.query("Compare Retrieval Augmented Generation (RAG) and Parameter efficient Finetuning (PEFT)")

In [20]:
res.response

'The provided context describes Retrieval-Augmented Generation (RAG) in detail but does not describe Parameter-Efficient Fine-Tuning (PEFT). Using only the given information, I can compare RAG to what is described about RAG itself; I cannot state specifics about PEFT because PEFT is not covered in the excerpts. Below is a comparison that highlights RAG’s characteristics and notes the missing information about PEFT.\n\nWhat RAG does (from the context)\n- Combines pretrained dense retrieval (a neural retriever over a dense vector index such as Wikipedia) with a pretrained seq2seq generator.\n- Uses non-parametric memory (retrieved documents / dense index) alongside parametric memory (the seq2seq model) so the generator can condition on up-to-date, explicit documents rather than only model parameters.\n- Has two formulation variants: one conditions on the same retrieved passages for the entire generated sequence; the other can use different retrieved passages per generated token.\n- Typic

In [21]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 40f472bb-265c-438a-bb45-25be7c023dc8
Title	 RAG
Text	 extractive downstream tasks. We explore ageneral-purpose fine-tuning recipe for retrieval-augmented generation (RAG) — models which combine pre-trainedparametric and non-parametric memory for language generation. We introduce RAG models where the parametric memory is apre-trained seq2seq model and the non-parametric memory is a dense vector index of Wikipedia, accessed with apre-trained neural retriever. We compare two RAG formulations, one which conditions on the same retrieved passagesacross the whole generated sequence, the other can use different passages per token. We fine-tune and evaluate ourmodels on a wide range of knowledge-intensive NLP tasks and set the state-of-the-art on three open domain QA tasks,outperforming parametric seq2seq models and task-specific retrieve-and-extract architectures. For language generationtasks, we find that RAG models generate more specific, diverse and factual language than a state-of

# Judger & Postprocessor — Simple Notes

## Judger Function

### What it is  
A **custom LLM judge** that scores retrieved nodes by semantic relevance to a query.

### What it does  
- Input:  
  - list of nodes (chunks)  
  - user query  
- LLM reads:
  - query  
  - all node texts  
- Outputs:
  - `node_id[]`  
  - `score[]` in range `[0, 1]`  

### Purpose  
Replace embedding similarity with **LLM reasoning-based relevance scoring**.

---

## OpenaiAsJudgePostprocessor

### What it is  
A **LlamaIndex NodePostprocessor** that uses the judger to rerank and filter nodes.

### Position in Pipeline

Retriever (top-k)

↓

OpenaiAsJudgePostprocessor

↓

Top-3 best nodes

↓

Generator



---

### What it does  

1. Calls `judger(nodes, query)`  
2. Gets:
   - node IDs  
   - relevance scores  
3. Sorts nodes by score (descending)  
4. Selects **top 3 nodes**  
5. Returns only those nodes  

---

## Why Use This

- Improves ranking accuracy  
- Ensures best chunks go to generator  
- Reduces hallucinations  
- Increases faithfulness  

---

## Trade-offs

- High accuracy  
- High latency  
- High cost  
- Not suitable for high-QPS serving  

---

## Key Takeaway

- **Judger** = LLM that scores chunk relevance  
- **Postprocessor** = integrates judger into RAG and keeps only best chunks  
- Together = **LLM-based reranking layer (RankGPT-style)** in LlamaIndex